# Hospitality Analytics - Silver to Gold (Business KPIs)

## 🎯 Purpose:
Transform cleaned silver layer data into actionable business intelligence KPIs for:
- Revenue Management Teams
- Operations Teams
- Marketing Teams
- Executive Leadership

## 📊 Gold Tables (KPIs):
1. **kpi_revpar** - Revenue Per Available Room (Industry standard)
2. **kpi_adr** - Average Daily Rate (Pricing metric)
3. **kpi_ancillary_attachment_rate** - Guest ancillary spend rate
4. **kpi_housekeeping_turnover_time** - Room cleaning efficiency
5. **kpi_weekend_vs_weekday_revenue** - Dynamic pricing validation

## 🏗️ Architecture:
```
SILVER (Clean Data)          GOLD (Business KPIs)
┌─────────────────────┐      ┌──────────────────┐
│ dim_guests          │──┐   │ kpi_revpar       │
│ fact_stays_unified  │──┼──▶│ kpi_adr          │
│ fact_room_avail_... │──┘   │ kpi_ancillary... │
└─────────────────────┘      │ kpi_housekeep... │
                             │ kpi_weekend_...  │
                             └──────────────────┘
```

**Author:** Data Engineering Team  
**Date:** January 2025  
**Shri Radhe Govind Ji! 🙏**

## Step 1: Environment Setup

In [0]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, sum as spark_sum, count, avg, min as spark_min, max as spark_max,
    round as spark_round, coalesce, lit, datediff, dayofweek, unix_timestamp,
    to_date, expr, countDistinct, percentile_approx
)
from pyspark.sql.window import Window

print("✅ Libraries imported successfully")

In [0]:
# Define configuration
CATALOG_NAME = 'hospitality_project'
SILVER_SCHEMA = 'silver_schema'
GOLD_SCHEMA = 'gold_schema'

print(f"📂 Configuration:")
print(f"  Silver Schema: {CATALOG_NAME}.{SILVER_SCHEMA}")
print(f"  Gold Schema: {CATALOG_NAME}.{GOLD_SCHEMA}")

## Step 2: Create Gold Schema

In [0]:
%sql
-- Create gold schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS hospitality_project.gold_schema;

SELECT 'Gold schema ready' AS status;

## Step 3: Load Silver Tables

In [0]:
print("\n📖 Loading Silver Tables...")

# Load silver tables
dim_guests = spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_guests")
fact_stays_unified = spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_stays_unified")
fact_room_availability = spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_room_availability_daily")

print(f"✅ dim_guests: {dim_guests.count():,} records")
print(f"✅ fact_stays_unified: {fact_stays_unified.count():,} records")
print(f"✅ fact_room_availability_daily: {fact_room_availability.count():,} records")

# Cache for performance (optional but recommended for multiple aggregations)
fact_stays_unified.cache()
fact_room_availability.cache()

print("\n💾 Tables cached for optimal performance")

## Step 4: KPI #1 - Revenue Per Available Room (RevPAR)

### Business Context:
**RevPAR** is THE gold standard metric in hospitality revenue management.

**Formula:** `Total Room Revenue / Total Available Rooms`

**Why it matters:**
- Combines occupancy rate and pricing power
- Industry benchmark for comparing hotel performance
- Helps decide: Should we raise prices or run promotions?

**Example:**
- Hotel has 100 rooms
- Sells 80 rooms at avg $150 = $12,000 revenue
- RevPAR = $12,000 / 100 = $120
- This means each room (available or not) generates $120/day

In [0]:
print("\n" + "="*80)
print("KPI #1: REVENUE PER AVAILABLE ROOM (RevPAR)")
print("="*80)

# Step 1: Calculate total room revenue per date and hotel
room_revenue_daily = (
    fact_stays_unified
    # Explode stays into daily revenue (allocate revenue across nights)
    .withColumn('date_array', 
                expr('sequence(check_in_date, date_sub(check_out_date, 1), interval 1 day)'))
    .withColumn('date', expr('explode(date_array)'))
    .withColumn('daily_room_revenue', 
                col('total_price') / col('stay_length_nights'))
    .groupBy('date', 'hotel_id')
    .agg(
        spark_sum('daily_room_revenue').alias('total_room_revenue')
    )
)

print(f"  Calculated daily room revenue: {room_revenue_daily.count():,} date-hotel records")

# Step 2: Count total available rooms per date and hotel
# Get unique rooms per hotel from availability table
total_rooms_daily = (
    fact_room_availability
    .groupBy('date', 'hotel_id')
    .agg(
        countDistinct('room_number').alias('total_available_rooms')
    )
)

print(f"  Calculated available rooms: {total_rooms_daily.count():,} date-hotel records")

# Step 3: Calculate RevPAR
kpi_revpar = (
    total_rooms_daily
    .join(room_revenue_daily, on=['date', 'hotel_id'], how='left')
    
    # Fill null revenue with 0 (days with no bookings)
    .withColumn('total_room_revenue', 
                coalesce(col('total_room_revenue'), lit(0.0)))
    
    # Calculate RevPAR
    .withColumn('revpar',
        when(col('total_available_rooms') > 0,
             spark_round(col('total_room_revenue') / col('total_available_rooms'), 2))
        .otherwise(lit(0.0)))
    
    .select(
        'date',
        'hotel_id',
        spark_round('total_room_revenue', 2).alias('total_room_revenue'),
        'total_available_rooms',
        'revpar'
    )
    .orderBy('date', 'hotel_id')
)

print(f"\n✅ RevPAR calculated: {kpi_revpar.count():,} records")

# Show sample
print("\nSample RevPAR data:")
kpi_revpar.show(10, truncate=False)

# Business insights
avg_revpar = kpi_revpar.select(avg('revpar')).first()[0]
print(f"\n📊 Average RevPAR across all hotels: ${avg_revpar:.2f}")
print("💡 Higher RevPAR = Better revenue management performance")

In [0]:
# Write to Delta
kpi_revpar.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('date') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_revpar")

print("✅ kpi_revpar table created successfully!")

## Step 5: KPI #2 - Average Daily Rate (ADR)

### Business Context:
**ADR** measures the average price per room sold (occupied rooms only).

**Formula:** `Total Room Revenue / Number of Rooms Sold`

**Why it matters:**
- Shows pricing effectiveness
- Helps identify if discounting too heavily
- Complements RevPAR (RevPAR = ADR × Occupancy Rate)

**Example:**
- Sold 80 rooms for $12,000 total
- ADR = $12,000 / 80 = $150 per sold room

**Insight:**
- High ADR + Low Occupancy = Prices too high
- Low ADR + High Occupancy = Leaving money on table

In [0]:
print("\n" + "="*80)
print("KPI #2: AVERAGE DAILY RATE (ADR)")
print("="*80)

# Calculate ADR per date and hotel
kpi_adr = (
    fact_stays_unified
    # Explode to daily level
    .withColumn('date_array', 
                expr('sequence(check_in_date, date_sub(check_out_date, 1), interval 1 day)'))
    .withColumn('date', expr('explode(date_array)'))
    .withColumn('daily_room_revenue', 
                col('total_price') / col('stay_length_nights'))
    
    .groupBy('date', 'hotel_id')
    .agg(
        spark_sum('daily_room_revenue').alias('total_room_revenue'),
        count('*').alias('rooms_sold')  # Each record = one room-night sold
    )
    
    # Calculate ADR
    .withColumn('adr',
        when(col('rooms_sold') > 0,
             spark_round(col('total_room_revenue') / col('rooms_sold'), 2))
        .otherwise(lit(0.0)))
    
    .select(
        'date',
        'hotel_id',
        spark_round('total_room_revenue', 2).alias('total_room_revenue'),
        'rooms_sold',
        'adr'
    )
    .orderBy('date', 'hotel_id')
)

print(f"\n✅ ADR calculated: {kpi_adr.count():,} records")

# Show sample
print("\nSample ADR data:")
kpi_adr.show(10, truncate=False)

# Business insights
avg_adr = kpi_adr.select(avg('adr')).first()[0]
print(f"\n📊 Average ADR across all hotels: ${avg_adr:.2f}")
print("💡 Track ADR trends to optimize pricing strategy")

In [0]:
# Write to Delta
kpi_adr.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('date') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_adr")

print("✅ kpi_adr table created successfully!")

## Step 6: KPI #3 - Ancillary Attachment Rate

### Business Context:
**Ancillary Attachment Rate** measures what % of guests spend money beyond their room.

**Formula:** `(Guests with POS spend / Total guests) × 100`

**Why it matters:**
- Reveals incremental revenue opportunities
- Helps marketing decide on package offers ("Free Breakfast" bundles)
- Identifies guest engagement with hotel amenities

**Example:**
- 100 guests stayed
- 65 used restaurant/spa/bar
- Attachment Rate = 65%

**Insight:**
- Low rate (<50%) = Need better promotion of amenities
- High rate (>70%) = Good guest engagement

In [0]:
print("\n" + "="*80)
print("KPI #3: ANCILLARY ATTACHMENT RATE")
print("="*80)

# Calculate attachment rate per date and hotel
kpi_ancillary = (
    fact_stays_unified
    # Explode to daily level
    .withColumn('date_array', 
                expr('sequence(check_in_date, date_sub(check_out_date, 1), interval 1 day)'))
    .withColumn('date', expr('explode(date_array)'))
    
    # Flag guests with ancillary spend
    .withColumn('has_ancillary_spend',
        when(col('pos_item_count') > 0, lit(1)).otherwise(lit(0)))
    
    .groupBy('date', 'hotel_id')
    .agg(
        count('*').alias('total_guests'),
        spark_sum('has_ancillary_spend').alias('guests_with_ancillary_spend'),
        spark_sum('total_pos_amount').alias('total_ancillary_revenue')
    )
    
    # Calculate attachment rate percentage
    .withColumn('attachment_rate_percentage',
        when(col('total_guests') > 0,
             spark_round((col('guests_with_ancillary_spend') / col('total_guests')) * 100, 2))
        .otherwise(lit(0.0)))
    
    # Calculate average ancillary spend per guest
    .withColumn('avg_ancillary_per_guest',
        when(col('total_guests') > 0,
             spark_round(col('total_ancillary_revenue') / col('total_guests'), 2))
        .otherwise(lit(0.0)))
    
    .select(
        'date',
        'hotel_id',
        'total_guests',
        'guests_with_ancillary_spend',
        'attachment_rate_percentage',
        spark_round('total_ancillary_revenue', 2).alias('total_ancillary_revenue'),
        'avg_ancillary_per_guest'
    )
    .orderBy('date', 'hotel_id')
)

print(f"\n✅ Ancillary attachment rate calculated: {kpi_ancillary.count():,} records")

# Show sample
print("\nSample Ancillary data:")
kpi_ancillary.show(10, truncate=False)

# Business insights
avg_attachment = kpi_ancillary.select(avg('attachment_rate_percentage')).first()[0]
avg_ancillary = kpi_ancillary.select(avg('avg_ancillary_per_guest')).first()[0]
print(f"\n📊 Average Attachment Rate: {avg_attachment:.1f}%")
print(f"📊 Average Ancillary Spend per Guest: ${avg_ancillary:.2f}")
print("💡 Target: >60% attachment rate for healthy F&B engagement")

In [0]:
# Write to Delta
kpi_ancillary.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('date') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_ancillary_attachment_rate")

print("✅ kpi_ancillary_attachment_rate table created successfully!")

## Step 7: KPI #4 - Housekeeping Turnover Time

### Business Context:
**Turnover Time** measures how quickly rooms are cleaned and ready after checkout.

**Formula:** `AVG(Time between checkout and 'Clean' status)`

**Why it matters:**
- Impacts ability to sell early check-ins
- Shows operational efficiency
- Helps optimize housekeeping schedules

**Example:**
- Guest checks out at 11:00 AM
- Room cleaned and marked 'Clean' at 1:30 PM
- Turnover time = 2.5 hours (150 minutes)

**Industry Standard:**
- Target: <2 hours for standard rooms
- <3 hours for suites

In [0]:
print("\n" + "="*80)
print("KPI #4: HOUSEKEEPING TURNOVER TIME")
print("="*80)

# Load housekeeping data
bronze_housekeeping = (
    spark.read.format("json")
    .load(f"/Volumes/{CATALOG_NAME}/bronze_schema/raw/housekeeping_logs")
    .filter(col('log_id').isNotNull())
    
    # Standardize status
    .withColumn('status',
        when(col('status').isNotNull(),
             when(upper(col('status')) == 'CLEAN', 'Clean')
             .when(upper(col('status')) == 'DIRTY', 'Dirty')
             .when(upper(col('status')) == 'MAINTENANCE', 'Maintenance')
             .when(upper(col('status')).contains('ORDER'), 'Out-of-Order')
             .otherwise('Unknown'))
        .otherwise('Unknown'))
    
    # Parse timestamp
    .withColumn('timestamp',
        coalesce(
            expr("try_to_timestamp(timestamp, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"),
            expr("try_to_timestamp(timestamp, 'dd/MM/yyyy HH:mm')"),
            expr("try_to_timestamp(timestamp, 'yyyy-MM-dd HH:mm:ss')")
        ))
    
    .filter(
        (col('status') == 'Clean') &
        (col('timestamp').isNotNull())
    )
    .select('hotel_id', 'room_number', 'status', 'timestamp')
)

print(f"  Loaded {bronze_housekeeping.count():,} 'Clean' status records")

# Get checkout events from fact_stays
# Need to read from bronze reservations to get room_number
checkouts = (
    spark.read.format("json")
    .load(f"/Volumes/{CATALOG_NAME}/bronze_schema/raw/reservations")
    .filter(col('res_id').isNotNull())
    .filter(col('room_number').isNotNull())
    
    # Parse dates
    .withColumn('check_out_date',
        coalesce(
            expr("try_to_date(check_out_date, 'yyyy-MM-dd')"),
            expr("try_to_date(check_out_date, 'dd/MM/yyyy')")
        ))
    
    .filter(col('check_out_date').isNotNull())
    
    # Assume checkout time is 11:00 AM (standard hotel checkout)
    .withColumn('checkout_timestamp',
        expr("to_timestamp(concat(check_out_date, ' 11:00:00'))"))
    
    .select(
        'res_id',
        'hotel_id',
        'room_number',
        'check_out_date',
        'checkout_timestamp'
    )
)

print(f"  Loaded {checkouts.count():,} checkout records")

# Join checkouts with housekeeping clean status
turnover_calc = (
    checkouts
    .join(
        bronze_housekeeping,
        on=['hotel_id', 'room_number'],
        how='inner'
    )
    
    # Only consider clean status on same day or next day after checkout
    .filter(
        (col('timestamp') >= col('checkout_timestamp')) &
        (col('timestamp') <= expr("checkout_timestamp + interval 1 day"))
    )
    
    # Calculate turnover time in minutes
    .withColumn('turnover_time_minutes',
        (unix_timestamp(col('timestamp')) - unix_timestamp(col('checkout_timestamp'))) / 60)
    
    # Only valid turnover times (positive and reasonable)
    .filter(
        (col('turnover_time_minutes') > 0) &
        (col('turnover_time_minutes') <= 600)  # Max 10 hours
    )
)

print(f"  Calculated {turnover_calc.count():,} valid turnover times")

# Aggregate by date and hotel
kpi_housekeeping = (
    turnover_calc
    .groupBy('check_out_date', 'hotel_id')
    .agg(
        count('*').alias('total_checkouts'),
        spark_round(avg('turnover_time_minutes'), 2).alias('avg_turnover_time_minutes'),
        spark_round(percentile_approx('turnover_time_minutes', 0.5), 2).alias('median_turnover_time_minutes'),
        spark_round(spark_min('turnover_time_minutes'), 2).alias('min_turnover_time_minutes'),
        spark_round(spark_max('turnover_time_minutes'), 2).alias('max_turnover_time_minutes')
    )
    .withColumnRenamed('check_out_date', 'date')
    .orderBy('date', 'hotel_id')
)

print(f"\n✅ Housekeeping turnover time calculated: {kpi_housekeeping.count():,} records")

# Show sample
print("\nSample Housekeeping Turnover data:")
kpi_housekeeping.show(10, truncate=False)

# Business insights
if kpi_housekeeping.count() > 0:
    avg_turnover = kpi_housekeeping.select(avg('avg_turnover_time_minutes')).first()[0]
    print(f"\n📊 Average Turnover Time: {avg_turnover:.0f} minutes ({avg_turnover/60:.1f} hours)")
    print("💡 Target: <120 minutes (2 hours) for optimal operations")
else:
    print("\n⚠️  No turnover data available (check housekeeping logs)")

In [0]:
# Write to Delta
kpi_housekeeping.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('date') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_housekeeping_turnover_time")

print("✅ kpi_housekeeping_turnover_time table created successfully!")

## Step 8: KPI #5 - Weekend vs Weekday Revenue (Dynamic Pricing Validation)

### Business Context:
**Weekend vs Weekday Analysis** validates dynamic pricing strategy.

**Metrics:**
- ADR by day type (weekend vs weekday)
- RevPAR by day type
- Occupancy Rate by day type

**Why it matters:**
- Validates 2x weekend pricing strategy from bronze data
- Identifies demand patterns
- Helps optimize pricing calendar

**Expected Pattern:**
- Weekend ADR should be ~2x weekday ADR
- Weekend occupancy may be higher (leisure travel)
- Overall weekend RevPAR should be significantly higher

In [0]:
print("\n" + "="*80)
print("KPI #5: WEEKEND VS WEEKDAY REVENUE")
print("="*80)

# Calculate weekend vs weekday metrics
kpi_weekend = (
    fact_stays_unified
    # Explode to daily level
    .withColumn('date_array', 
                expr('sequence(check_in_date, date_sub(check_out_date, 1), interval 1 day)'))
    .withColumn('date', expr('explode(date_array)'))
    .withColumn('daily_room_revenue', 
                col('total_price') / col('stay_length_nights'))
    
    # Identify weekend (Saturday=7, Sunday=1 in dayofweek)
    .withColumn('day_of_week', dayofweek(col('date')))
    .withColumn('is_weekend',
        when(col('day_of_week').isin([1, 7]), lit(True))
        .otherwise(lit(False)))
    
    # Get total available rooms for occupancy calc
    .join(
        fact_room_availability.groupBy('date', 'hotel_id')
        .agg(countDistinct('room_number').alias('total_rooms')),
        on=['date', 'hotel_id'],
        how='left'
    )
    
    .groupBy('date', 'hotel_id', 'is_weekend')
    .agg(
        spark_sum('daily_room_revenue').alias('total_room_revenue'),
        count('*').alias('rooms_sold'),
        spark_max('total_rooms').alias('total_available_rooms')
    )
    
    # Calculate metrics
    .withColumn('adr',
        when(col('rooms_sold') > 0,
             spark_round(col('total_room_revenue') / col('rooms_sold'), 2))
        .otherwise(lit(0.0)))
    
    .withColumn('occupancy_rate',
        when(col('total_available_rooms') > 0,
             spark_round((col('rooms_sold') / col('total_available_rooms')) * 100, 2))
        .otherwise(lit(0.0)))
    
    .withColumn('revpar',
        when(col('total_available_rooms') > 0,
             spark_round(col('total_room_revenue') / col('total_available_rooms'), 2))
        .otherwise(lit(0.0)))
    
    .select(
        'date',
        'hotel_id',
        'is_weekend',
        'adr',
        'revpar',
        'occupancy_rate',
        'rooms_sold',
        'total_available_rooms',
        spark_round('total_room_revenue', 2).alias('total_room_revenue')
    )
    .orderBy('date', 'hotel_id')
)

print(f"\n✅ Weekend vs Weekday metrics calculated: {kpi_weekend.count():,} records")

# Show sample
print("\nSample Weekend vs Weekday data:")
kpi_weekend.show(10, truncate=False)

# Business insights - Compare weekend vs weekday
weekend_stats = kpi_weekend.filter(col('is_weekend')).select(
    avg('adr').alias('weekend_adr'),
    avg('occupancy_rate').alias('weekend_occ')
).first()

weekday_stats = kpi_weekend.filter(~col('is_weekend')).select(
    avg('adr').alias('weekday_adr'),
    avg('occupancy_rate').alias('weekday_occ')
).first()

if weekend_stats and weekday_stats:
    weekend_adr = weekend_stats['weekend_adr'] or 0
    weekday_adr = weekday_stats['weekday_adr'] or 1  # Avoid division by zero
    
    price_premium = (weekend_adr / weekday_adr - 1) * 100 if weekday_adr > 0 else 0
    
    print(f"\n📊 Weekend vs Weekday Analysis:")
    print(f"  Weekend ADR: ${weekend_adr:.2f}")
    print(f"  Weekday ADR: ${weekday_adr:.2f}")
    print(f"  Weekend Premium: {price_premium:.1f}%")
    print(f"  Weekend Occupancy: {weekend_stats['weekend_occ']:.1f}%")
    print(f"  Weekday Occupancy: {weekday_stats['weekday_occ']:.1f}%")
    print(f"\n💡 Expected: ~100% weekend premium (2x pricing)")
    
    if abs(price_premium - 100) < 10:
        print("✅ Dynamic pricing strategy validated!")
    else:
        print("⚠️  Weekend premium differs from expected 2x pricing")

In [0]:
# Write to Delta
kpi_weekend.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('date') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_weekend_vs_weekday_revenue")

print("✅ kpi_weekend_vs_weekday_revenue table created successfully!")

## Step 9: Final Gold Layer Summary & Validation

In [0]:
print("\n" + "="*80)
print("GOLD LAYER - FINAL SUMMARY")
print("="*80)

# List all gold tables
gold_tables = [
    'kpi_revpar',
    'kpi_adr',
    'kpi_ancillary_attachment_rate',
    'kpi_housekeeping_turnover_time',
    'kpi_weekend_vs_weekday_revenue'
]

print("\n📊 GOLD LAYER TABLES CREATED:")
for table in gold_tables:
    try:
        df = spark.table(f"{CATALOG_NAME}.{GOLD_SCHEMA}.{table}")
        count = df.count()
        print(f"  ✅ {table}: {count:,} records")
    except:
        print(f"  ❌ {table}: Not found")

print("\n🎯 BUSINESS VALUE:")
print("  📈 kpi_revpar → Revenue management decisions")
print("  💰 kpi_adr → Pricing strategy optimization")
print("  🍽️  kpi_ancillary_attachment_rate → F&B upsell opportunities")
print("  🧹 kpi_housekeeping_turnover_time → Operations efficiency")
print("  📅 kpi_weekend_vs_weekday_revenue → Dynamic pricing validation")

print("\n✅ GOLD LAYER COMPLETE!")
print("🎉 End-to-End Pipeline: Bronze → Silver → Gold SUCCESSFUL!")
print("="*80)

In [0]:
# Optional: Create a master KPI dashboard view
print("\n📊 Creating Unified KPI Dashboard View...")

# Join key metrics for executive dashboard
kpi_dashboard = (
    kpi_revpar.select(
        col('date'),
        col('hotel_id'),
        col('revpar')
    )
    .join(
        kpi_adr.select(col('date'), col('hotel_id'), col('adr')),
        on=['date', 'hotel_id'],
        how='left'
    )
    .join(
        kpi_ancillary.select(
            col('date'), 
            col('hotel_id'), 
            col('attachment_rate_percentage'),
            col('avg_ancillary_per_guest')
        ),
        on=['date', 'hotel_id'],
        how='left'
    )
    .orderBy('date', 'hotel_id')
)

print("\nExecutive Dashboard Sample:")
kpi_dashboard.show(10, truncate=False)

# Write dashboard view
kpi_dashboard.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.kpi_executive_dashboard")

print("\n✅ Executive Dashboard view created!")

## 📝 Documentation & Next Steps

### ✅ What We Accomplished:

**Gold Layer Tables Created:**
1. ✅ `kpi_revpar` - Industry standard revenue metric
2. ✅ `kpi_adr` - Pricing effectiveness measure
3. ✅ `kpi_ancillary_attachment_rate` - Upsell performance
4. ✅ `kpi_housekeeping_turnover_time` - Operational efficiency
5. ✅ `kpi_weekend_vs_weekday_revenue` - Pricing strategy validation
6. ✅ `kpi_executive_dashboard` - Unified view for leadership

### 🎯 Business Use Cases:

**Revenue Management Team:**
- Monitor RevPAR trends to optimize pricing
- Compare ADR across hotels to identify outliers
- Validate weekend pricing premium

**Operations Team:**
- Track housekeeping efficiency
- Optimize staffing based on turnover times
- Identify bottlenecks in room readiness

**Marketing Team:**
- Measure ancillary attachment rates
- Design packages to increase F&B revenue
- Promote amenities with low attachment

**Executive Leadership:**
- Single dashboard with all key metrics
- Track performance across hotel portfolio
- Make data-driven strategic decisions

### 📊 Data Pipeline Architecture:

```
BRONZE (Raw)     →     SILVER (Clean)     →     GOLD (KPIs)
────────────          ────────────────         ─────────────
JSON Files            Delta Tables             Aggregated Metrics
Data Quality Issues   SCD Type 2              Business Intelligence
Auto Loader           Validated Data           Executive Dashboards
Schema Evolution      Deduplication            Revenue Management
```

### 🚀 Ready for:
- ✅ BI Tool Integration (Tableau, Power BI, Looker)
- ✅ Automated Reporting
- ✅ Real-time Dashboards
- ✅ Machine Learning (Forecasting, Pricing Optimization)

---

**🎉 PROJECT COMPLETE! 🎉**

**Shri Radhe Govind Ji! 🙏**